In [1]:
import glob
import itertools
import polars as pl
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["pdf.use14corefonts"] = True
rcParams["text.usetex"] = False  # not use LaTeX

In [2]:
CHR_ORDER = [f"chr{i}" for i in itertools.chain(range(1, 23), ("X", "Y"))]
CHR_COLORS = [
    "#403E80",
    "#2C477D",
    "#2C477D",
    "#346B9B",
    "#3C80AA",
    "#4587A2",
    "#589D96",
    "#73ACA4",
    "#87BAAF",
    "#94BE9F",
    "#9FC38C",
    "#A1C27C",
    "#A4C165",
    "#C2C969",
    "#B5A957",
    "#DFD06C",
    "#F2D46C",
    "#E3C765",
    "#E5BA61",
    "#D89E56",
    "#C8824A",
    "#BB6B3E",
    "#AB5C40",
    "#9B4C41",
    "#8C3C42",
    "#731F37",
]
CHR_PALETTE = dict(zip(CHR_ORDER, CHR_COLORS))

In [3]:
dfs_all_cdr_chrom: list[pl.DataFrame] = []
for chrom in CHR_ORDER:
    for files in sorted(glob.glob(f"data/hgsvc/cdr_distribution/{chrom}_*.xls")):
        if isinstance(files, str):
            files = [files]
        for file in files:
            name_arr = file.replace(f"data/hgsvc/cdr_distribution/{chrom}_", "").replace(".cdr.xls", "")
            df_cdr_chrom = pl.read_csv(file, separator="\t", has_header=True).with_columns(chrom=pl.lit(chrom), arr=pl.lit(name_arr))
            dfs_all_cdr_chrom.append(df_cdr_chrom)

df_all_cdr_chrom: pl.DataFrame = pl.concat(dfs_all_cdr_chrom).cast({"chrom": pl.Enum(CHR_ORDER), "arr": pl.Int8}).with_columns(pl.col("arr") - 1)
df_all_cdr_chrom

index,count,chrom,arr
i64,i64,enum,i8
0,0,"""chr1""",0
1,0,"""chr1""",0
2,0,"""chr1""",0
3,1,"""chr1""",0
4,1,"""chr1""",0
…,…,…,…
95,0,"""chrY""",0
96,0,"""chrY""",0
97,0,"""chrY""",0


In [4]:
df_arr_window_type = pl.read_csv("data/hgsvc/array_window_type.bed", separator="\t", new_columns=["chrom", "arr", "index", "label"]).cast({"chrom": pl.Enum(CHR_ORDER), "arr": pl.Int8}).with_columns(pl.col("arr") - 1)
df_arr_window_type

chrom,arr,index,label
enum,i8,i64,str
"""chr10""",0,1,"""live"""
"""chr10""",0,2,"""live"""
"""chr10""",0,3,"""live"""
"""chr10""",0,4,"""live"""
"""chr10""",0,5,"""live"""
…,…,…,…
"""chrY""",0,95,"""live"""
"""chrY""",0,96,"""live"""
"""chrY""",0,97,"""live"""


In [5]:
df_all_cdr_chrom_join = df_all_cdr_chrom.join(df_arr_window_type, on=["chrom", "arr", "index"], how="left")
df_all_cdr_chrom_join

index,count,chrom,arr,label
i64,i64,enum,i8,str
0,0,"""chr1""",0,"""live"""
1,0,"""chr1""",0,"""live"""
2,0,"""chr1""",0,"""live"""
3,1,"""chr1""",0,"""live"""
4,1,"""chr1""",0,"""live"""
…,…,…,…,…
95,0,"""chrY""",0,"""live"""
96,0,"""chrY""",0,"""live"""
97,0,"""chrY""",0,"""live"""


In [6]:
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages("plot/all_chrs.pdf") as pdf:
    chr_to_idx = dict((c, i) for i, c in enumerate(CHR_ORDER))
    for grp, df_grp in df_all_cdr_chrom_join.group_by(["chrom"], maintain_order=True):
        chrom = grp[0]

        # Remove lt 3 per array.
        srs_remove_arr = df_grp.group_by(["arr"]).agg(pl.col("count").sum()).filter(pl.col("count") < 3).get_column("arr")
        df_grp = df_grp.filter(~pl.col("arr").is_in(srs_remove_arr))
        num_arrs = df_grp["arr"].n_unique()
        print(chrom, num_arrs)
        # Only create n subplots.
        fig, axes = plt.subplots(
            1, num_arrs, figsize=(10, 3),
            sharey=True
        )
        if num_arrs == 1:
            ax_iter = [axes]
        else:
            ax_iter = axes

        for ((arr, df_grp_arr), ax) in zip(df_grp.group_by(["arr"], maintain_order=True), ax_iter):
            arr = arr[0]
            arr = arr + 1
            print(chrom, 1, arr)
            ax: plt.Axes

            # Get continguous live region coordinates.
            df_live_idxs = (
                df_grp_arr
                .with_columns(ctg_grp=pl.col("label").rle_id())
                .group_by(["ctg_grp"])
                .agg(
                    label=pl.col("label").first(),
                    st=pl.col("index").min(),
                    end=pl.col("index").max()
                )
                .filter(pl.col("label") == "live")
            )

            # Bar for each index position and count.
            ax.bar(
                df_grp_arr["index"], df_grp_arr["count"], color=CHR_COLORS[chr_to_idx[chrom]]
            )

            # Add red live regions.
            for _, _, st, end in df_live_idxs.iter_rows():
                ax.axvspan(
                    st, end + 1, ymin=0, ymax=100, facecolor="red", alpha=0.1, label="Live"
                )
            
            # ({df_grp_arr["count"].sum()})
            ax.set_title(f"{chrom} - {arr} Array(s)")

            # Set limits.
            ax.set_xlim(0, 100)

            ax.grid(False)
            ax.set_facecolor("white")
            # Draw spines of plot
            for spine in ["left", "bottom"]:
                ax.spines[spine].set_color("k")
            for spine in ["right", "top"]:
                ax.spines[spine].set_color(None)
            # Add ticks.
            ax.tick_params("both", left=True, bottom=True)
        
        # Remove duplicate entries.
        # https://stackoverflow.com/a/13589144
        handles, labels = plt.gca().get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax_iter[-1].legend(
            by_label.values(),
            by_label.keys(),
            loc="upper right",
            bbox_to_anchor=(1.0, 1.0),
            fancybox=True
        )

        fig.supxlabel("Relative Position")
        fig.supylabel("Count")
        fig.subplots_adjust(top=0.5)
        fig.tight_layout()
        pdf.savefig()
        fig.savefig(f"plot/{chrom}.pdf")
        fig.clear()
        fig

chr1 3
chr1 1 1
chr1 1 2
chr1 1 3
chr2 1
chr2 1 1
chr3 2
chr3 1 2
chr3 1 3
chr4 5
chr4 1 1
chr4 1 2
chr4 1 3
chr4 1 4
chr4 1 5
chr5 1
chr5 1 1
chr6 1
chr6 1 1
chr7 2
chr7 1 1
chr7 1 2
chr8 1
chr8 1 1
chr9 1
chr9 1 1
chr10 1
chr10 1 1
chr11 1
chr11 1 1
chr12 2
chr12 1 1
chr12 1 2
chr13 1
chr13 1 1
chr14 1
chr14 1 1
chr15 1
chr15 1 1
chr16 1
chr16 1 1
chr17 2
chr17 1 1
chr17 1 2
chr18 1
chr18 1 1
chr19 2
chr19 1 1
chr19 1 2
chr20 2
chr20 1 1
chr20 1 2
chr21 1
chr21 1 1


/tmp/ipykernel_48274/1472855463.py:14: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(


chr22 1
chr22 1 1
chrX 1
chrX 1 1
chrY 1
chrY 1 1


<Figure size 1000x300 with 0 Axes>

<Figure size 1000x300 with 0 Axes>

...

<Figure size 1000x300 with 0 Axes>

<Figure size 1000x300 with 0 Axes>

<Figure size 1000x300 with 0 Axes>

<Figure size 1000x300 with 0 Axes>

<Figure size 1000x300 with 0 Axes>